# Step 3: Load Cleaned Data into SQLite
Create schema with constraints, load cleaned CSVs, verify row counts and relationships.

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('../data/ecommerce.db')
cur = conn.cursor()

with open('../sql/schema.sql') as f:
    cur.executescript(f.read())

conn.commit()
print('schema created')


schema created


In [2]:
customers_df = pd.read_csv('../data/cleaned/customers_clean.csv')
products_df = pd.read_csv('../data/cleaned/products_clean.csv')
orders_df = pd.read_csv('../data/cleaned/orders_clean.csv')
order_items_df = pd.read_csv('../data/cleaned/order_items_clean.csv')

customers_df.to_sql('customers', conn, if_exists='append', index=False)
products_df.to_sql('products', conn, if_exists='append', index=False)
orders_df.to_sql('orders', conn, if_exists='append', index=False)
order_items_df.to_sql('order_items', conn, if_exists='append', index=False)

conn.commit()
print('data loaded')


data loaded


## Verify row counts

In [3]:
for table in ['customers', 'products', 'orders', 'order_items']:
    count = pd.read_sql(f'SELECT COUNT(*) as n FROM {table}', conn)['n'][0]
    print(table, count)


customers 185
products 60
orders 728
order_items 1874


## Verify relationships (no orphan rows)

In [4]:
check1 = pd.read_sql("""
SELECT COUNT(*) as bad_orders FROM orders o
LEFT JOIN customers c ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL
""", conn)

check2 = pd.read_sql("""
SELECT COUNT(*) as bad_items FROM order_items oi
LEFT JOIN orders o ON oi.order_id = o.order_id
WHERE o.order_id IS NULL
""", conn)

print(check1)
print(check2)


   bad_orders
0           0
   bad_items
0          0


In [5]:
conn.close()
print('connection closed')


connection closed
